# Filling in the days: the numbers and the figures

Companion notebook to the GeoAI Watch post of November 2026. [Open it in Colab](https://colab.research.google.com/github/heykuldip/geoaiwatch/blob/2026-11-gap-filling/posts/2026-11-gap-filling/gap_filling.ipynb).

1. **The numbers.** Every number in the post that comes from the results, and figures 1, 2 and 4, from the run's tables in `runs/`. A few seconds, with no imagery.
2. **The example frame** (off by default). The example frame's quality file and image, read from the TUM record over FTP without downloading the archive whole, and figure 3.

Code: MIT. Results and figures: CC BY-SA 4.0. Text: CC BY-NC-ND 4.0.

In [ ]:
import os
import sys
from pathlib import Path

REF = "2026-11-gap-filling"  # this notebook's release; "main" tries unreleased code

if "google.colab" in sys.modules:
    import subprocess

    repo = Path(f"/content/geoaiwatch-{REF}")             # one folder for each release
    if not repo.exists():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", REF,
                        "https://github.com/heykuldip/geoaiwatch", str(repo)], check=True)

    def git(*args):
        return subprocess.run(["git", "-C", str(repo), *args], capture_output=True,
                              text=True).stdout.strip()

    head = git("rev-parse", "HEAD")
    if not head or head != git("rev-parse", f"{REF}^{{commit}}") or \
            git("status", "--porcelain", "--untracked-files=no"):
        raise SystemExit(f"{repo} isn't a clean checkout of {REF}: delete the folder, or "
                         "restart the runtime, and run this cell again")
    print(f"code and results at {REF}, commit {head[:7]}")
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)

here = Path.cwd()
POST = here / "posts" / "2026-11-gap-filling"
if (here / "scripts" / "series_figures.py").exists():
    POST = here
ROOT = POST.parents[1]
os.environ.setdefault("DEN_HOME", str(ROOT / "data" / "DynamicEarthNet"))
MODULES = ("geoaiwatch", "series_plan", "series_qa", "series_run", "series_figures",
           "fetch_example", "fetch_frame")
for name in [m for m in sys.modules if m.split(".")[0] in MODULES]:
    del sys.modules[name]
# a running kernel can't see a package installed after it started, so use the source directly
sys.path[:0] = [str(ROOT / "src"), str(POST / "scripts")]

## 1. The numbers

Each row is a number the post gives, or the value behind its words ("most", "within a week"), computed from the run's tables. Shares are rounded as the post rounds them: whole percentages, with the decimals a share needs where it would otherwise read 0% or 100%, and a share that serves as a lower bound rounded down. Where the post gives a value for each place, the places run Renshou, Fengcheng, Yangzhou, Andong.

In [ ]:
import json
from fractions import Fraction

import pandas as pd
import series_figures as F
import series_plan as SP
import series_qa as Q

design, _ = SP.load()                     # the frozen plan, checked against its SHA-256
RUNS = POST / "runs"
PLACES = [p.name for p in design.places]
TWO_YEARS = f"{design.days[0]:%Y-%m}..{design.days[-1]:%Y-%m}"


def read(name):
    return pd.read_csv(RUNS / name)


def by_place(df):
    return df.set_index("place").loc[PLACES]


def each(num, den):
    return ", ".join(F.pct(int(n), int(d)) for n, d in zip(num, den))


def listed(values):
    return ", ".join(str(int(v)) for v in values)


def table(rows):
    return pd.DataFrame(rows, columns=["in the post", "value"])


pd.set_option("display.max_colwidth", None)
monthly = read("qa_monthly.csv")
years = by_place(monthly[monthly.period == TWO_YEARS])
frames = read("label_frames.csv")
codes = read("label_codes.csv")
audit = json.loads((RUNS / "audit.json").read_text())
post1 = pd.read_parquet(POST.parent / "2026-11-near-infrared" / "artifacts"
                        / "cell_qa_status.parquet")

### Two years, day by day

In [ ]:
on_actual = [d["l2_codes_on_fully_actual"] for d in audit["diagnostics"].values()]
s2_l8 = sum(c.get("900", 0) + c.get("-900", 0) for c in on_actual)
gaps = read("gaps.csv")
pixel = gaps[gaps.level == "pixel"]
by_day, by_gap = (by_place(pixel[pixel.weight == w]) for w in ("pixel_days", "gaps"))
checked = audit["inputs"]["metadata"].values()


rows = [
    ("days in a row at each place", listed(years.n_days)),
    ("the last post's test tiles entirely filled in",
     F.pct(int(post1.fully_synthetic_cell.sum()), len(post1))),
    ("fully-actual pixel-days with a same-day Sentinel-2 or Landsat 8 code",
     F.pct(s2_l8, sum(sum(c.values()) for c in on_actual))),
    ("pixel-days fully actual, the four places",
     F.pct(int(years.n_fully_actual.sum()), int(years.n_pixel_days.sum()))),
    ("pixel-days fully actual, each place", each(years.n_fully_actual, years.n_pixel_days)),
    (f"days with every pixel fully actual, of {design.n_days}", listed(years.days_all)),
    ("days with no pixel fully actual", each(years.days_none, years.n_days)),
    ("median gap, for a filled-in pixel-day, days", listed(by_day["median"])),
    ("median gap, each gap once, days", listed(by_gap["median"])),
    ("longest gap, days", listed(by_gap["max"])),
    ("files checked for embedded fill dates; with any",
     f"{len(checked)}; {sum(bool(v['beyond_image_structure']) for v in checked)}"),
]
table(rows)

### The map dates

In [ ]:
pixels, filled = int(frames.n_pixels.sum()), int(frames.n_filled.sum())
later = int(frames.n_l2_future.sum())
all_1, all_100 = int(frames.l1_all_1.sum()), int(frames.l1_all_100.sum())
nov = post1.groupby(["cube", "date"])[["fully_observed_cell", "fully_synthetic_cell"]].all()


def offsets(side):
    """Median and largest distance in days, weighted by filled pixels."""
    c = codes[codes.l2.between(1, 899) if side == "later" else codes.l2.between(-899, -1)]
    d, w = c.l2.abs(), c.n_filled
    return Q.quantile(d, w, 5), int(d.max())


later_shares = sorted(Fraction(int(n), int(d)) for n, d in zip(frames.n_l2_future,
                                                              frames.n_pixels))
median = later_shares[(len(later_shares) + 1) // 2 - 1]     # the run's median, inverted CDF
(e50, emax), (l50, lmax) = offsets("earlier"), offsets("later")
rows = [
    ("days with maps at each place; images in all",
     f"{listed(frames.groupby('place').size().loc[PLACES])}; {len(frames)}"),
    ("pixels fully actual; filled in",
     f"{F.pct(int(frames.n_fully_actual.sum()), pixels)}; {F.pct(filled, pixels)}"),
    ("images fully actual in every pixel; entirely filled in; a mix",
     f"{all_1}; {all_100}; {len(frames) - all_1 - all_100}"),
    ("the same two in the last post's records",
     f"{int(nov.fully_observed_cell.sum())}; {int(nov.fully_synthetic_cell.sum())}"),
    ("filled pixels with a later recorded source, at least", F.pct_at_least(later, filled)),
    ("all the pixels filled in with a later source, at least", F.pct_at_least(later, pixels)),
    ("median offset, earlier and later sources, days", f"{e50}; {l50}"),
    ("the farthest, days back and ahead", f"{emax}; {lmax}"),
    ("images with some pixels filled in with a later source",
     f"{int((frames.n_l2_future > 0).sum())} of {len(frames)}"),
    ("images with every pixel filled in with a later source",
     str(int((frames.n_l2_future == frames.n_pixels).sum()))),
    ("the median image's pixels filled in with a later source",
     F.pct(median.numerator, median.denominator)),
    ("images with none", str(int((frames.n_l2_future == 0).sum()))),
]
table(rows)

### The example frame, from the tables

The image the plan's rule picks: the largest share of pixels filled in with a later Layer 2 source, the earliest date among ties. Part 2 reads the image itself.

In [ ]:
top = max(Fraction(int(n), int(d)) for n, d in zip(frames.n_l2_future, frames.n_pixels))
tied = frames[[Fraction(int(n), int(d)) == top
               for n, d in zip(frames.n_l2_future, frames.n_pixels)]]
pick = tied.sort_values(["date", "cube"]).iloc[0]
here_codes = codes[(codes.cube == pick.cube) & (codes.date == pick.date)]
later_here = here_codes[here_codes.l2.between(1, 899)]
rows = [
    ("the image", f"{pick.place}, {pick.date}"),
    ("its pixels entirely filled in", F.pct(int(pick.n_entirely_filled), int(pick.n_pixels))),
    ("its pixels with a later source", F.pct(int(pick.n_l2_future), int(pick.n_pixels))),
    ("their sources, days later", f"{int(later_here.l2.min())} to {int(later_here.l2.max())}"),
    ("nine in ten within, days", str(Q.quantile(later_here.l2, later_here.n_filled, 9))),
    ("images at that share", str(len(tied))),
    ("the first day of the series", str(design.days[0])),
]
table(rows)

### Real days before each map

In [ ]:
summary = read("windows_summary.csv")
own, common = summary[summary.dates == "own"], summary[summary.dates == "common"]


def window(rows, name):
    return by_place(rows[rows.window == name])


d15, d30, d60 = (window(own, f"nested_{n}") for n in (15, 30, 60))
windows = read("windows.csv")
complete = windows[windows.complete]
every = windows.groupby(["cube", "date"]).complete.all()
dates = sorted(set(every[every].index.get_level_values("date")))
first = complete.groupby("window").date.min()
rows = [
    ("map dates on which every window is complete; the first; the last",
     f"{len(dates)}; {dates[0]}; {dates[-1]}"),
    ("quarters of an image at each place on those dates",
     listed(window(common, "nested_15").n_cells)),
    ("complete quarters at each place, 15 days; from", f"{listed(d15.n_cells)}; {first.nested_15}"),
    ("complete quarters at each place, 30 days; from", f"{listed(d30.n_cells)}; {first.nested_30}"),
    ("complete quarters at each place, 60 days; from", f"{listed(d60.n_cells)}; {first.nested_60}"),
    ("pixels with a fully-actual day in the 15 days before", each(d15.n_covered, d15.n_pixels)),
    ("pixels with a fully-actual day in the 30 days before", each(d30.n_covered, d30.n_pixels)),
    ("pixels with a fully-actual day in the 60 days before", each(d60.n_covered, d60.n_pixels)),
    ("the median pixel's fully-actual days in the 30 days before", listed(d30.days_p50)),
]
table(rows)

### Figures 1, 2 and 4

Drawn by the same code as the post's figures, from the same tables. Each number a figure prints is checked against a second table first; a failed check stops the cell.

In [ ]:
import tempfile

import matplotlib as mpl
from IPython.display import Image, display

t, _ = F.read_runs(RUNS)
FIGDIR = Path(tempfile.mkdtemp())
drawn = {F.FIGS[0]: (F.draw_strips, F.daily_strips(design, t, F.CAPTIONS)),
         F.FIGS[1]: (F.draw_bars, F.label_bars(design, t, F.CAPTIONS)),
         F.FIGS[3]: (F.draw_windows, F.window_cells(design, t, F.CAPTIONS))}
with mpl.rc_context(F.STYLE):
    for name, (draw, shown) in drawn.items():
        draw(design, shown, FIGDIR / name)
for name in drawn:
    display(Image(filename=str(FIGDIR / name), width=700))

## 2. The example frame and figure 3 (optional)

Set `GET_FRAME = True` and run the cell. It reads the example frame's quality file and image from inside the TUM archive over FTP, without downloading the archive whole, checks each against the SHA-256 the run recorded before writing it under `DEN_HOME`, and draws figure 3, checking what it prints against the tables. Files already under `DEN_HOME` with the recorded SHA-256 aren't fetched again.

In [ ]:
GET_FRAME = False

if GET_FRAME:
    import fetch_example
    from geoaiwatch import config

    assert fetch_example.main([]) == 0, "the fetch stopped; its message is above"
    frame = F.example_frame(design, t, config.DEN_HOME, F.CAPTIONS)
    place = next(p.name for p in design.places if p.cube == frame["cube"])
    with mpl.rc_context(F.STYLE):
        F.draw_frame(design, frame, place, FIGDIR / F.FIGS[2])
    shown = frame["printed"]
    display(table([
        ("the image", f"{place}, {frame['date']}"),
        ("every pixel entirely filled in, read from its quality file",
         "yes" if shown["entirely_filled"] else "no"),
        ("its pixels with a later source", shown["share"]),
        ("their sources, days later", f"{shown['offsets'][0]} to {shown['offsets'][1]}"),
    ]))
    display(Image(filename=str(FIGDIR / F.FIGS[2]), width=700))